# Conformalized BNE — Simulation Prototype

**Goal:** Compare conformal prediction intervals vs BNE credible intervals on 2D simulation (Mishra's Bird function).

**Methods:**
1. BNE Credible Intervals (baseline)
2. Split Conformal on BNE mean
3. CQR on BNE quantiles
4. **Spatial CQR** (our contribution)

**Metrics:** Marginal coverage, conditional coverage (in-domain vs OOD), average interval width, NLL

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Add parent paths so we can import wrapper_functions and conformal
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../case_study_results'))

from wrapper_functions import (
    generate_data_2d, run_base_models, run_bma_model, run_bne_model,
    make_bma_samples, make_bne_samples
)
from conformal.conformal_bne import (
    split_conformal, conformalized_quantile_regression, spatial_cqr,
    bne_credible_interval, extract_bne_predictions, compute_cdf_at_obs
)
from conformal.metrics import (
    marginal_coverage, conditional_coverage, average_interval_width,
    interval_width_by_group, coverage_by_quantile, negative_log_likelihood
)

print("All imports successful.")

## 0. Setup & Imports

In [ ]:
## Sanity check: conformalize a known N(0,1)
np.random.seed(42)
n_sanity = 5000
y_sanity = np.random.randn(n_sanity)

# Split: first half calibration, second half test
n_half = n_sanity // 2
y_cal_s = y_sanity[:n_half]
y_test_s = y_sanity[n_half:]

# "Prediction" is just the mean = 0
pred_cal = np.zeros(n_half)
pred_test = np.zeros(n_sanity - n_half)

for alpha, label in [(0.05, '95%'), (0.10, '90%')]:
    lo, hi = split_conformal(y_cal_s, pred_cal, pred_test, alpha=alpha)
    cov = marginal_coverage(y_test_s, lo, hi)
    width = average_interval_width(lo, hi)
    print(f"Split CP ({label}): coverage = {cov:.4f}, width = {width:.4f}")

    # CQR with known quantiles
    from scipy.stats import norm
    q_lo_cal = norm.ppf(alpha / 2) * np.ones(n_half)
    q_hi_cal = norm.ppf(1 - alpha / 2) * np.ones(n_half)
    q_lo_test = norm.ppf(alpha / 2) * np.ones(n_sanity - n_half)
    q_hi_test = norm.ppf(1 - alpha / 2) * np.ones(n_sanity - n_half)

    lo_cqr, hi_cqr = conformalized_quantile_regression(
        y_cal_s,
        np.stack([q_lo_cal, q_hi_cal], axis=1),
        np.stack([q_lo_test, q_hi_test], axis=1),
        alpha=alpha
    )
    cov_cqr = marginal_coverage(y_test_s, lo_cqr, hi_cqr)
    width_cqr = average_interval_width(lo_cqr, hi_cqr)
    print(f"CQR      ({label}): coverage = {cov_cqr:.4f}, width = {width_cqr:.4f}")
    print()

## 0.5 Sanity Check: Conformalize N(0,1)

Verify that split conformal achieves exact 95% coverage on known Gaussian data.

In [ ]:
## Configuration
SEED = 42
N_TOTAL_TRAIN = 500   # Total points for train + calibration
N_BASE = 100
N_TEST = 2000
ALPHA = 0.05          # 95% target coverage

np.random.seed(SEED)

# Generate data using existing function
X_base, X_train_full, X_test, Y_base, Y_train_full, Y_test, mean_test = \
    generate_data_2d(N_train=N_TOTAL_TRAIN, N_base=N_BASE, N_test=N_TEST, seed=SEED)

print(f"X_base: {X_base.shape}, Y_base: {Y_base.shape}")
print(f"X_train_full: {X_train_full.shape}, Y_train_full: {Y_train_full.shape}")
print(f"X_test: {X_test.shape}, Y_test: {Y_test.shape}")

# Split train_full into train (70%) and calibration (30%)
n_full = len(X_train_full)
n_train = int(n_full * 0.7)
n_cal = n_full - n_train

# Shuffle indices
idx = np.random.permutation(n_full)
train_idx = idx[:n_train]
cal_idx = idx[n_train:]

X_train = X_train_full[train_idx]
Y_train = Y_train_full[train_idx]
X_cal = X_train_full[cal_idx]
Y_cal = Y_train_full[cal_idx]

print(f"\nAfter split:")
print(f"  Train: {X_train.shape[0]} points")
print(f"  Calibration: {X_cal.shape[0]} points")
print(f"  Test: {X_test.shape[0]} points")

# Label test points as in-domain vs OOD
# Training region: [-pi, pi]^2, test region: [-1.25*pi, 1.25*pi]^2
in_domain_mask = (np.abs(X_test[:, 0]) <= np.pi) & (np.abs(X_test[:, 1]) <= np.pi)
ood_mask = ~in_domain_mask

test_region = np.where(in_domain_mask, 'in-domain', 'OOD')
print(f"  Test in-domain: {in_domain_mask.sum()}, OOD: {ood_mask.sum()}")

## 1. Generate 2D Simulation Data

Using Mishra's Bird function. We generate data and split into train/calibration/test.

In [ ]:
%%time
# Combine calibration + test for prediction (we'll separate after)
X_pred = np.concatenate([X_cal, X_test], axis=0)
Y_pred = np.concatenate([Y_cal, Y_test], axis=0)

# 2a. Train base models
base_preds_train, base_preds_pred, kernel_names = run_base_models(
    X_base, X_train, X_pred, Y_base, Y_train, Y_pred
)
print(f"Base models trained: {kernel_names}")
print(f"base_preds_train: {base_preds_train.shape}")
print(f"base_preds_pred: {base_preds_pred.shape}")

In [ ]:
%%time
# 2b. Train BMA
bma_joint_samples, X_train_mcmc, Y_train_mcmc, means_train_mcmc, means_pred_mcmc = \
    run_bma_model(
        X_train, X_pred, Y_train,
        base_preds_train, base_preds_pred,
        gp_lengthscale=1.,
        gp_l2_regularizer=0.1,
        y_noise_std=0.1,
        map_step_size=0.1,
        map_num_steps=10_000,
        mcmc_step_size=0.1,
        mcmc_num_steps=10_000,
        mcmc_nchain=10,
        mcmc_burnin=2_500,
        mcmc_initialize_from_map=True,
        n_samples_eval=1000,
        n_samples_train=100,
        n_samples_test=200,
        return_mcmc_examples=True,
        seed=SEED,
    )

print(f"BMA done.")
print(f"  means_train_mcmc: {means_train_mcmc.shape}")
print(f"  means_pred_mcmc: {means_pred_mcmc.shape}")

In [ ]:
%%time
# 2c. Train BNE (variance mode — heterogeneous variance)
bne_samples_dict = run_bne_model(
    X_train=X_train_mcmc,
    Y_train=Y_train_mcmc,
    X_test=X_pred,
    base_model_samples_train=means_train_mcmc,
    base_model_samples_test=means_pred_mcmc,
    moment_mode='variance',
    gp_lengthscale=1.,
    gp_l2_regularizer=10.,
    map_step_size=5e-3,
    map_num_steps=10_000,
    mcmc_step_size=1e-2,
    mcmc_num_steps=10_000,
    mcmc_burnin=2_500,
    mcmc_nchain=10,
    mcmc_initialize_from_map=True,
    seed=SEED,
)

print(f"BNE done.")
print(f"  Keys: {list(bne_samples_dict.keys())}")
for k, v in bne_samples_dict.items():
    if hasattr(v, 'shape'):
        print(f"  {k}: {v.shape}")

In [ ]:
# Extract BNE predictions on the combined pred set
bne_preds = extract_bne_predictions(bne_samples_dict, alpha=ALPHA)

# Split back into calibration and test portions
# X_pred = [X_cal (n_cal), X_test (N_TEST)]
bne_mean_cal = bne_preds['mean'][:n_cal]
bne_mean_test = bne_preds['mean'][n_cal:]
bne_quantiles_cal = bne_preds['quantiles'][:n_cal]
bne_quantiles_test = bne_preds['quantiles'][n_cal:]
bne_samples_cal = bne_preds['samples'][:, :n_cal]
bne_samples_test = bne_preds['samples'][:, n_cal:]

print(f"BNE posterior samples shape: {bne_preds['samples'].shape}")
print(f"  Cal mean: {bne_mean_cal.shape}, Test mean: {bne_mean_test.shape}")
print(f"  Cal quantiles: {bne_quantiles_cal.shape}, Test quantiles: {bne_quantiles_test.shape}")

## 3. Extract Predictions & Split into Calibration / Test

In [ ]:
# Method 1: BNE Credible Intervals (no conformal correction)
lo_credible, hi_credible = bne_credible_interval(bne_samples_test, alpha=ALPHA)

# Method 2: Split Conformal on BNE mean
lo_split, hi_split = split_conformal(Y_cal, bne_mean_cal, bne_mean_test, alpha=ALPHA)

# Method 3: CQR on BNE quantiles
lo_cqr, hi_cqr = conformalized_quantile_regression(
    Y_cal, bne_quantiles_cal, bne_quantiles_test, alpha=ALPHA
)

# Method 4: Spatial CQR (our contribution) — try multiple lengthscales
spatial_results = {}
for ls in [0.5, 1.0, 2.0, 5.0]:
    lo_s, hi_s = spatial_cqr(
        Y_cal, bne_quantiles_cal, bne_quantiles_test,
        X_cal, X_test,
        alpha=ALPHA, kernel='matern32', lengthscale=ls
    )
    spatial_results[ls] = (lo_s, hi_s)

# Use ls=2.0 as default spatial CQR
lo_spatial, hi_spatial = spatial_results[2.0]

print("All prediction intervals computed.")

## 4. Compute All Prediction Intervals

In [ ]:
# Build results table
methods = {
    'BNE Credible': (lo_credible, hi_credible),
    'Split CP': (lo_split, hi_split),
    'CQR': (lo_cqr, hi_cqr),
    'Spatial CQR (l=2)': (lo_spatial, hi_spatial),
}

rows = []
for name, (lo, hi) in methods.items():
    # Overall
    cov_all = marginal_coverage(Y_test, lo, hi)
    width_all = average_interval_width(lo, hi)

    # Conditional: in-domain vs OOD
    cov_cond = conditional_coverage(Y_test, lo, hi, test_region)
    width_cond = interval_width_by_group(lo, hi, test_region)

    cov_in = cov_cond.get('in-domain', np.nan)
    cov_ood = cov_cond.get('OOD', np.nan)
    w_in = width_cond.get('in-domain', np.nan)
    w_ood = width_cond.get('OOD', np.nan)

    rows.append({
        'Method': name,
        'Coverage (all)': f'{cov_all:.3f}',
        'Coverage (in-domain)': f'{cov_in:.3f}',
        'Coverage (OOD)': f'{cov_ood:.3f}',
        'Width (all)': f'{width_all:.3f}',
        'Width (in-domain)': f'{w_in:.3f}',
        'Width (OOD)': f'{w_ood:.3f}',
    })

# Add spatial CQR with other lengthscales
for ls, (lo_s, hi_s) in spatial_results.items():
    if ls == 2.0:
        continue
    cov_all = marginal_coverage(Y_test, lo_s, hi_s)
    width_all = average_interval_width(lo_s, hi_s)
    cov_cond = conditional_coverage(Y_test, lo_s, hi_s, test_region)
    width_cond = interval_width_by_group(lo_s, hi_s, test_region)
    rows.append({
        'Method': f'Spatial CQR (l={ls})',
        'Coverage (all)': f'{cov_all:.3f}',
        'Coverage (in-domain)': f'{cov_cond.get("in-domain", np.nan):.3f}',
        'Coverage (OOD)': f'{cov_cond.get("OOD", np.nan):.3f}',
        'Width (all)': f'{width_all:.3f}',
        'Width (in-domain)': f'{width_cond.get("in-domain", np.nan):.3f}',
        'Width (OOD)': f'{width_cond.get("OOD", np.nan):.3f}',
    })

df_results = pd.DataFrame(rows)
print(f"Target coverage: {1 - ALPHA:.0%}\n")
print(df_results.to_string(index=False))

## 5. Evaluation: Coverage Table

In [ ]:
# Compute CDF at observed values for BNE posterior
cdf_vals = compute_cdf_at_obs(bne_samples_test, Y_test)
cal_diag = coverage_by_quantile(Y_test, cdf_vals, n_bins=20)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(cal_diag['expected'], cal_diag['observed'], 'b.-', label=f'BNE (ECE={cal_diag["ece"]:.3f})')
ax.set_xlabel('Expected cumulative probability')
ax.set_ylabel('Observed cumulative probability')
ax.set_title('BNE Posterior Calibration (PIT)')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 6. Calibration Diagnostics (ECE-style)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

plot_methods = {
    'BNE Credible': (lo_credible, hi_credible),
    'Split CP': (lo_split, hi_split),
    'CQR': (lo_cqr, hi_cqr),
    'Spatial CQR (l=2)': (lo_spatial, hi_spatial),
}

for ax, (name, (lo, hi)) in zip(axes.ravel(), plot_methods.items()):
    widths = hi - lo
    covered = (Y_test >= lo) & (Y_test <= hi)

    sc = ax.scatter(X_test[:, 0], X_test[:, 1], c=widths, s=3,
                    cmap='viridis', alpha=0.7)
    plt.colorbar(sc, ax=ax, label='Interval width')

    # Mark training region boundary
    rect = plt.Rectangle((-np.pi, -np.pi), 2*np.pi, 2*np.pi,
                         fill=False, edgecolor='red', linewidth=1.5, linestyle='--')
    ax.add_patch(rect)

    cov = marginal_coverage(Y_test, lo, hi)
    avg_w = average_interval_width(lo, hi)
    ax.set_title(f'{name}\nCov={cov:.3f}, Width={avg_w:.2f}')
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')

plt.suptitle(f'Interval Width Maps (target: {1-ALPHA:.0%} coverage)', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 7. Spatial Visualization

Compare interval widths across 2D space for each method.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, (name, (lo, hi)) in zip(axes.ravel(), plot_methods.items()):
    covered = (Y_test >= lo) & (Y_test <= hi)

    ax.scatter(X_test[covered, 0], X_test[covered, 1], c='steelblue', s=3,
               alpha=0.4, label='Covered')
    ax.scatter(X_test[~covered, 0], X_test[~covered, 1], c='red', s=10,
               alpha=0.8, label='Missed', marker='x')

    rect = plt.Rectangle((-np.pi, -np.pi), 2*np.pi, 2*np.pi,
                         fill=False, edgecolor='black', linewidth=1.5, linestyle='--',
                         label='Training region')
    ax.add_patch(rect)

    cov = marginal_coverage(Y_test, lo, hi)
    ax.set_title(f'{name} (Coverage={cov:.3f})')
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.legend(loc='lower right', fontsize=8)

plt.suptitle('Coverage Maps: Blue=covered, Red=missed', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 8. Coverage Map: Covered vs Missed Points

In [ ]:
# Lengthscale sensitivity analysis for Spatial CQR
ls_values = [0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0]
ls_rows = []

for ls in ls_values:
    lo_s, hi_s = spatial_cqr(
        Y_cal, bne_quantiles_cal, bne_quantiles_test,
        X_cal, X_test,
        alpha=ALPHA, kernel='matern32', lengthscale=ls
    )
    cov = marginal_coverage(Y_test, lo_s, hi_s)
    width = average_interval_width(lo_s, hi_s)
    cov_cond = conditional_coverage(Y_test, lo_s, hi_s, test_region)
    ls_rows.append({
        'lengthscale': ls,
        'coverage': cov,
        'width': width,
        'cov_in_domain': cov_cond.get('in-domain', np.nan),
        'cov_ood': cov_cond.get('OOD', np.nan),
    })

df_ls = pd.DataFrame(ls_rows)
print(df_ls.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(df_ls['lengthscale'], df_ls['coverage'], 'bo-', label='Overall')
ax1.plot(df_ls['lengthscale'], df_ls['cov_in_domain'], 'gs-', label='In-domain')
ax1.plot(df_ls['lengthscale'], df_ls['cov_ood'], 'r^-', label='OOD')
ax1.axhline(1 - ALPHA, color='k', linestyle='--', alpha=0.5, label=f'Target ({1-ALPHA:.0%})')
ax1.set_xlabel('Lengthscale')
ax1.set_ylabel('Coverage')
ax1.set_title('Spatial CQR: Coverage vs Lengthscale')
ax1.legend()
ax1.set_xscale('log')

ax2.plot(df_ls['lengthscale'], df_ls['width'], 'ko-')
ax2.set_xlabel('Lengthscale')
ax2.set_ylabel('Average Interval Width')
ax2.set_title('Spatial CQR: Width vs Lengthscale')
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

## 9. Spatial CQR Lengthscale Sensitivity

In [ ]:
# NLL from BNE posterior (same for all conformal methods — conformal only adjusts intervals)
nll = negative_log_likelihood(Y_test, bne_samples_test)
print(f"BNE Posterior NLL: {nll:.4f}")

## 10. NLL Comparison

In [ ]:
# NOTE: This cell takes a long time to run (~3x the single run above).
# Uncomment and run when ready for full comparison.

"""
sample_sizes = [250, 500, 1000]
size_results = []

for n_total in sample_sizes:
    print(f"\n{'='*60}")
    print(f"Running with N_train = {n_total}")
    print(f"{'='*60}")

    # Generate data
    Xb, Xtr_full, Xte, Yb, Ytr_full, Yte, mte = \
        generate_data_2d(N_train=n_total, N_base=100, N_test=2000, seed=SEED)

    # Split
    nf = len(Xtr_full)
    nt = int(nf * 0.7)
    ix = np.random.RandomState(SEED).permutation(nf)
    Xtr, Ytr = Xtr_full[ix[:nt]], Ytr_full[ix[:nt]]
    Xc, Yc = Xtr_full[ix[nt:]], Ytr_full[ix[nt:]]
    nc = len(Xc)

    Xp = np.concatenate([Xc, Xte])

    # Base models
    bp_tr, bp_p, kn = run_base_models(Xb, Xtr, Xp, Yb, Ytr, np.concatenate([Yc, Yte]))

    # BMA
    _, Xt_mcmc, Yt_mcmc, mt_mcmc, mp_mcmc = run_bma_model(
        Xtr, Xp, Ytr, bp_tr, bp_p,
        mcmc_initialize_from_map=True, return_mcmc_examples=True, seed=SEED)

    # BNE
    bne_s = run_bne_model(
        Xt_mcmc, Yt_mcmc, Xp, mt_mcmc, mp_mcmc,
        moment_mode='variance', mcmc_initialize_from_map=True, seed=SEED)

    preds = extract_bne_predictions(bne_s, alpha=ALPHA)

    # Split into cal/test
    mn_c, mn_t = preds['mean'][:nc], preds['mean'][nc:]
    qt_c, qt_t = preds['quantiles'][:nc], preds['quantiles'][nc:]

    # Regions
    in_dom = (np.abs(Xte[:, 0]) <= np.pi) & (np.abs(Xte[:, 1]) <= np.pi)
    regions = np.where(in_dom, 'in-domain', 'OOD')

    for method_name, (lo, hi) in [
        ('BNE Credible', bne_credible_interval(preds['samples'][:, nc:], ALPHA)),
        ('Split CP', split_conformal(Yc, mn_c, mn_t, ALPHA)),
        ('CQR', conformalized_quantile_regression(Yc, qt_c, qt_t, ALPHA)),
        ('Spatial CQR', spatial_cqr(Yc, qt_c, qt_t, Xc, Xte, ALPHA, lengthscale=2.0)),
    ]:
        cov = marginal_coverage(Yte, lo, hi)
        w = average_interval_width(lo, hi)
        cc = conditional_coverage(Yte, lo, hi, regions)
        size_results.append({
            'N': n_total, 'Method': method_name,
            'Coverage': f'{cov:.3f}', 'Width': f'{w:.3f}',
            'Cov_InDomain': f'{cc.get("in-domain", 0):.3f}',
            'Cov_OOD': f'{cc.get("OOD", 0):.3f}',
        })

print("\\n" + "="*60)
print(pd.DataFrame(size_results).to_string(index=False))
"""
print("Cell commented out — uncomment to run full sample size comparison.")

## 11. Sample Size Sensitivity (n=250, 500, 1000)

Quick comparison across different training sizes.

## 2. Train Base Models + BMA + BNE

Reuse existing pipeline. Train on train split, predict on calibration + test.